# Deep Learning 007 — What a Perceptron Cannot Do

AND, OR and NAND are easy. XOR is impossible — and *impossible* here means proved, not
"we tried a while". This notebook fits the three that work, then attacks XOR with
everything and fails on purpose.

In [ ]:
import numpy as np
import itertools
import matplotlib.pyplot as plt

G = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
GATES = {'AND':  np.array([0, 0, 0, 1]),
         'OR':   np.array([0, 1, 1, 1]),
         'NAND': np.array([1, 1, 1, 0]),
         'XOR':  np.array([0, 1, 1, 0])}

def step(z):
    return (z >= 0).astype(int)

def fit(X, y, lr=0.1, epochs=500, seed=0):
    rng = np.random.default_rng(seed)
    w, b = rng.normal(size=2) * 0.1, 0.0
    for _ in range(epochs):
        for i in rng.permutation(len(X)):
            err = y[i] - step(X[i] @ w + b)
            w = w + lr * err * X[i]
            b = b + lr * err
    return w, b, (step(X @ w + b) == y).mean()

In [ ]:
for name, y in GATES.items():
    w, b, acc = fit(G, y)
    print(f'{name:5} accuracy {acc:.0%}   w={np.round(w,2)} b={b:.2f}')

Three gates at 100%. XOR sits at 50% or 75% depending on the seed — never 100%.

## Attack XOR properly

More epochs, every learning rate, a hundred random starts. If it were merely difficult,
*something* would work.

In [ ]:
best = 0.0
results = []
for lr in (0.001, 0.01, 0.1, 0.5, 1.0):
    for epochs in (100, 1000, 5000):
        accs = [fit(G, GATES['XOR'], lr=lr, epochs=epochs, seed=s)[2]
                for s in range(20)]
        results.append((lr, epochs, max(accs)))
        best = max(best, max(accs))
print(f"{'lr':>7}{'epochs':>9}{'best of 20 seeds':>19}")
for lr, ep, a in results:
    print(f'{lr:>7}{ep:>9}{a:>19.0%}')
print(f'\nbest accuracy over {len(results)*20} runs: {best:.0%}')

## Why — and this is a proof, not a shrug

XOR needs `(0,0)→0`, `(1,1)→0`, `(0,1)→1`, `(1,0)→1`. A perceptron computes
`step(w·x + b)`, so its decision regions are the two sides of a straight line. Write out
the four required inequalities and add two of them together:

- `(0,0)→0` requires $b < 0$
- `(1,1)→0` requires $w_1 + w_2 + b < 0$
- `(0,1)→1` requires $w_2 + b \ge 0$
- `(1,0)→1` requires $w_1 + b \ge 0$

Adding the last two: $w_1 + w_2 + 2b \ge 0$. Combined with $b < 0$ that gives
$w_1 + w_2 > -2b > 0$, hence $w_1 + w_2 + b > -b > 0$ — which **contradicts** the second
requirement. No `(w, b)` exists. Let's also confirm it by brute force over a fine grid.

In [ ]:
grid = np.linspace(-4, 4, 61)
found = None
for w1, w2, b in itertools.product(grid, grid, grid):
    if (step(G @ np.array([w1, w2]) + b) == GATES['XOR']).all():
        found = (w1, w2, b); break
print('exhaustive search over 61^3 =', 61**3, 'parameter triples')
print('a perfect XOR line:', found)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
for a, (name, y) in zip(ax, GATES.items()):
    w, b, acc = fit(G, y)
    a.scatter(G[y == 0, 0], G[y == 0, 1], marker='o', s=90, label='0')
    a.scatter(G[y == 1, 0], G[y == 1, 1], marker='^', s=90, label='1')
    if abs(w[1]) > 1e-9:
        xs = np.linspace(-0.5, 1.5, 20)
        a.plot(xs, -(w[0] / w[1]) * xs - b / w[1], 'k-')
    a.set(title=f'{name}  {acc:.0%}', xlim=(-0.5, 1.5), ylim=(-0.5, 1.5))
plt.tight_layout(); plt.show()

Look at the XOR panel. The two `1`s are on opposite corners, with the two `0`s on the
other diagonal. **No single straight line separates one diagonal from the other.**

More data does not help, because the problem is not a shortage of data. More epochs do
not help, because the algorithm is not failing to converge — there is nothing to
converge to. **The model class is wrong**, and the fix is a second layer, which is
lesson 009.

## Exercises

1. Add more copies of the four XOR points (with tiny noise) and re-run. Confirm that
   thousands of rows change nothing.
2. Add a **third** input `x1*x2` and fit a perceptron on all three columns. It now
   works — explain why, and why that is feature engineering rather than deep learning.
3. Which of the 16 possible two-input boolean gates are learnable by one perceptron?
   Enumerate them and count.